#DAY 6 Databricks Challenge

###We use 3 layers of Data Model - Bronze, Silver and Gold

####Task 1 - Bronze (Reading and appending only. No deletes or updates) - Raw Ingestion

In [0]:
from pyspark.sql import functions as F

# Read raw CSV from the correct raw_data volume
raw = spark.read.csv(
    "/Volumes/workspace/default/raw_data/2019-Oct.csv",
    header=True,
    inferSchema=True
)
#Ensure the file exists in the specified path
# Add ingestion timestamp
bronze = raw.withColumn("ingestion_ts", F.current_timestamp())

# Write to Bronze layer inside challenge volume
bronze.write \
    .format("delta") \
    .mode("append") \
    .save("/Volumes/workspace/default/challenge/bronze_events")


In [0]:
display(bronze)

#**************************************************

####Task 2 - Silver(Data quality applied, deduplication and snapshot created) – Cleaned & Validated Data

In [0]:
bronze = spark.read.format("delta") \
    .load("/Volumes/workspace/default/challenge/bronze_events")

silver = (
    bronze
    .filter(F.col("price") > 0)
    .filter(F.col("price") < 10000)
    .dropDuplicates(["user_session", "event_time"])
    .withColumn("event_date", F.to_date("event_time"))
    .withColumn(
        "price_tier",
        F.when(F.col("price") < 10, "budget")
         .when(F.col("price") < 50, "mid")
         .otherwise("premium")
    )
)

silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/challenge/silver_events")


In [0]:
display(silver)

#**************************************************

####Task 3 - Gold(Business Ready Metrics) – Business Aggregates

In [0]:
silver = spark.read.format("delta") \
    .load("/Volumes/workspace/default/challenge/silver_events")

product_perf = (
    silver
    .groupBy("product_id")
    .agg(
        F.countDistinct(
            F.when(F.col("event_type") == "view", F.col("user_id"))
        ).alias("views"),

        F.countDistinct(
            F.when(F.col("event_type") == "purchase", F.col("user_id"))
        ).alias("purchases"),

        F.sum(
            F.when(F.col("event_type") == "purchase", F.col("price"))
        ).alias("revenue")
    )
    .withColumn(
        "conversion_rate",
        F.when(F.col("views") > 0,
               F.col("purchases") / F.col("views") * 100)
         .otherwise(0)
    )
)

product_perf.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/challenge/gold_products")


In [0]:
display(product_perf)

#**************************************************

## **For me more such learning and insights in**
- ### [LinkedIn](https://www.linkedin.com/in/ilakkiyan-av/) 
- ### [Youtube](https://www.youtube.com/@ilakkiyanav) 